## Gemini: Connecting to Gemini

The Gemini API allows you to connect to Google's most powerful multi-modal model. This example configures your API key and sends an example message to the API and prints a response.

Before you start, visit https://aistudio.google.com/app/apikey to create an API key.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
import google.generativeai as genai
from google.colab import userdata

# Configure the API key
genai.configure(api_key="xxxxxxxxxxxxxxxxxxxxxxxxxxxxx")

You are using models that cost money!  However, Google allows for some free usage of their models per day.  So, if you get blocked because you reached your limit (rate limitted), simply change the model to some other model.  Below, I have included the code that lists all the availble models, and here is link to the rate limits:  https://ai.google.dev/gemini-api/docs/rate-limits.
Older models allow for more free access than newer models.  Stick to base Gemini models for text generation.

In [ ]:
for model in genai.list_models():
  print(model.name)

In [4]:
## you can use the lastest model, but you get only so many free tokens
## flash light is an older model, and allows more requests
## see: https://ai.google.dev/gemini-api/docs/rate-limits
model = genai.GenerativeModel('gemini-2.0-flash-lite')

In [ ]:
prompt = "What is the capital of France?"
response = model.generate_content(prompt)
print(response.text)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
file_path = '/content/drive/My Drive/Colab Notebooks/finance_data.csv' # Update with the correct path if needed

try:
    with open(file_path, 'r') as f:
        csv_string = f.read()
    print("File content read successfully!")
    print(csv_string[:500]) # Print the first 500 characters to show the content
except FileNotFoundError:
    print(f"File not found at {file_path}. Please check the path and try again.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
prompt = f"Analyze the following financial data provided in CSV format:\n\n{csv_string}\n\nBased on this data, provide insights into trends, key metrics, and any notable observations."
print("Prompt generated:")
print(prompt[:500] + "...") # Print the beginning of the prompt to show its structure

In [ ]:
# Send the prompt to the model for analysis
# Note: This will consume your API quota. Please ensure you have addressed any quota issues (e.g., TooManyRequests error) before running this cell.
try:
    response = model.generate_content(prompt)
    print("Analysis from Gemini model:")
    print(response.text)
except Exception as e:
    print(f"An error occurred while calling the model: {e}")

In [ ]:
# Generate a new prompt specifically for stock price prediction
predict_prompt = f"Analyze the following financial data provided in CSV format and attempt to predict the stock price for each entry based on the other financial metrics. Please provide your predicted stock prices and compare that with the given price:\n\n{csv_string}.  Generate output in CSV format.  Output only the csv data and nothing else"
print("Prediction prompt generated:")
print(predict_prompt[:500] + "...") # Print the beginning of the prompt to show its structure

In [ ]:
# Send the prediction prompt to the model
# Note: This will consume your API quota. Please ensure you have addressed any quota issues (e.g., TooManyRequests error) before running this cell.
try:
    response = model.generate_content(predict_prompt)
    print("Prediction attempt from Gemini model:")
    print(response.text)
except Exception as e:
    print(f"An error occurred while calling the model: {e}")

Here you can utilize Gemini to help you code.   You want to convert this CSV format to somethin you can further analyze, graph, etc.   Ask Gemini something like: "convert reponse.text to dataframe".  It will generate the following code for you.   
You can also further instruct it to remove any leading text: "remove the leading explanation text as well. only the data should be in df_predicted"
You want to end up with only the data.   


In [ ]:
import pandas as pd
import io

# Assuming response.text from the previous cell contains the CSV data
csv_data_string = response.text

# Remove the leading explanation line if it exists
explanation_line = "Prediction attempt from Gemini model:\n"
if csv_data_string.startswith(explanation_line):
    csv_data_string = csv_data_string[len(explanation_line):]

# Remove the leading "```csv\n" and trailing "```" if they exist
if csv_data_string.startswith("```csv\n"):
    csv_data_string = csv_data_string[len("```csv\n"):]
if csv_data_string.endswith("```"):
    csv_data_string = csv_data_string[:-len("```")]

# Read the CSV data from the string into a pandas DataFrame
df_predicted = pd.read_csv(io.StringIO(csv_data_string))

print("DataFrame created from model response:")
display(df_predicted.head())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get unique company IDs
company_ids = df_predicted['company_id'].unique()

# Determine the number of plots and grid dimensions
n_companies = len(company_ids)
n_cols = 3 # Number of columns for the grid of plots
n_rows = (n_companies + n_cols - 1) // n_cols # Calculate number of rows

# Create subplots
fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, n_rows * 5))
axes = axes.flatten() # Flatten the axes array for easy iteration

# Plot for each company
for i, company_id in enumerate(company_ids):
    # Filter data for the current company
    company_data = df_predicted[df_predicted['company_id'] == company_id]

    # Plot actual and predicted stock prices
    axes[i].plot(company_data['stock_price'], label='Actual Stock Price')
    axes[i].plot(company_data['predicted_stock_price'], label='Predicted Stock Price')

    axes[i].set_title(f'Stock Price Prediction for {company_id}')
    axes[i].set_xlabel('Data Point Index') # Using index as x-axis for now
    axes[i].set_ylabel('Stock Price')
    axes[i].legend()
    axes[i].grid(True)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()